In [1]:
import requests
import pytz
from datetime import datetime
import time
session = requests.Session()
def weather(city):
    s = time.time()
    
    cache = {
        "guntur": (16.30, 80.43),
        "vizag": (17.68, 83.21),
        "narasaraopet": (16.23, 80.05),
        "hyderabad": (17.38, 78.48),
        "vijayawada": (16.50, 80.64)
    }
    
    name = city.strip().lower()
    
    if name in cache:
        lat, lon = cache[name]
        src = "Cache"
    else:
           geourl = f"https://nominatim.openstreetmap.org/search?q={name}&format=json&limit=1"
           geodata = session.get(geourl, headers={'User-Agent': 'TurboApp'}).json()
           lat, lon = geodata[0]['lat'], geodata[0]['lon']
           src = "API"

    apiurl = (
        f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}"
        "&current_weather=true&daily=temperature_2m_max,temperature_2m_min,sunrise,sunset&timezone=auto"
    )
    res = session.get(apiurl).json()

    curr = res['current_weather']
    daily = res['daily']
    tz_name = res['timezone']
    
    local_tz = pytz.timezone(tz_name)
    now = datetime.now(local_tz)

    return {
        "Location": city.title(),
        "Time": now.strftime('%I:%M %p'),
        "Temp": f"{curr['temperature']}°C",
        "Max/Min": f"{daily['temperature_2m_max'][0]} / {daily['temperature_2m_min'][0]}°C",
        "Wind": f"{curr['windspeed']} km/h",
        "Sun": f"↑{daily['sunrise'][0][-5:]} ↓{daily['sunset'][0][-5:]}",
        "Latency": f"{time.time() - s:.4f}s ({src})"
    }

city = input("City: ")
data = weather(city)

print("\n" + "═"*40)
for k, v in data.items():
    print(f" {k:10} | {v}")
print("═"*40)

City:  goa



════════════════════════════════════════
 Location   | Goa
 Time       | 12:21 AM
 Temp       | 27.9°C
 Max/Min    | 33.0 / 25.8°C
 Wind       | 1.1 km/h
 Sun        | ↑06:18 ↓18:48
 Latency    | 1.4396s (API)
════════════════════════════════════════
